# 🚀 YOLO-UDD v2.0 - FAST Training (10x Faster!)

**⚡ OPTIMIZED: COCO conversion in ~5-10 minutes instead of 4 hours!**

Repository: https://github.com/kshitijkhede/YOLO-UDD-v2.0

## Step 1: Clone Repository

In [ ]:
!git clone https://github.com/kshitijkhede/YOLO-UDD-v2.0.git
%cd YOLO-UDD-v2.0
print("✅ Repository cloned!")

## Step 2: Install Dependencies

In [ ]:
!pip install -q torch torchvision albumentations opencv-python-headless tqdm pyyaml tensorboard
print("✅ Dependencies installed!")

## Step 3: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

## Step 4: Check Dataset

In [ ]:
import os

DATASET_ROOT = '/content/drive/MyDrive/TrashCan_dataset'
ORIGINAL_DATA = os.path.join(DATASET_ROOT, 'original_data')
IMAGES_DIR = os.path.join(ORIGINAL_DATA, 'images')
ANNOTATIONS_DIR = os.path.join(ORIGINAL_DATA, 'annotations')

print(f"Images: {IMAGES_DIR}")
print(f"Annotations: {ANNOTATIONS_DIR}")

if os.path.exists(IMAGES_DIR):
    img_count = len([f for f in os.listdir(IMAGES_DIR) if f.endswith('.jpg')])
    print(f"✅ {img_count} images found")
else:
    print("❌ Images not found!")

if os.path.exists(ANNOTATIONS_DIR):
    ann_count = len([f for f in os.listdir(ANNOTATIONS_DIR) if f.endswith('.json')])
    print(f"✅ {ann_count} annotations found")
else:
    print("❌ Annotations not found!")

## Step 5: ⚡ FAST COCO Conversion

**KEY OPTIMIZATION:** Gets image dimensions from JSON (not opening 7,212 image files!)

In [ ]:
import json
import random
from tqdm import tqdm
import time

def fast_coco_convert(images_dir, annotations_dir, output_file, split='train', ratio=0.8):
    coco = {
        "images": [],
        "annotations": [],
        "categories": [
            {"id": 1, "name": "trash"},
            {"id": 2, "name": "animal"},
            {"id": 3, "name": "rov"}
        ]
    }
    
    # Get image list
    images = sorted([f for f in os.listdir(images_dir) if f.endswith('.jpg')])
    random.seed(42)
    random.shuffle(images)
    
    split_idx = int(len(images) * ratio)
    selected = images[:split_idx] if split == 'train' else images[split_idx:]
    
    ann_id = 1
    
    for img_id, img_file in enumerate(tqdm(selected, desc=f"{split} set"), 1):
        ann_file = os.path.join(annotations_dir, img_file + '.json')
        
        if not os.path.exists(ann_file):
            continue
        
        try:
            with open(ann_file, 'r') as f:
                data = json.load(f)
            
            # Get size from JSON (FAST!)
            size = data.get('size', {'width': 1920, 'height': 1080})
            
            coco["images"].append({
                "id": img_id,
                "file_name": img_file,
                "width": size.get('width', 1920),
                "height": size.get('height', 1080)
            })
            
            # Process annotations
            for obj in data.get('objects', []):
                cls = obj.get('classTitle', '').lower()
                cat_id = 1  # default trash
                if 'animal' in cls or 'fish' in cls:
                    cat_id = 2
                elif 'rov' in cls or 'robot' in cls:
                    cat_id = 3
                
                if 'points' in obj and 'exterior' in obj['points']:
                    pts = obj['points']['exterior']
                    xs = [p[0] for p in pts]
                    ys = [p[1] for p in pts]
                    
                    x1, x2 = min(xs), max(xs)
                    y1, y2 = min(ys), max(ys)
                    w, h = x2 - x1, y2 - y1
                    
                    if w > 0 and h > 0:
                        coco["annotations"].append({
                            "id": ann_id,
                            "image_id": img_id,
                            "category_id": cat_id,
                            "bbox": [x1, y1, w, h],
                            "area": w * h,
                            "iscrowd": 0
                        })
                        ann_id += 1
        except:
            pass
    
    with open(output_file, 'w') as f:
        json.dump(coco, f)
    
    print(f"✅ {split}: {len(coco['images'])} images, {len(coco['annotations'])} boxes")
    return coco

print("⚡ Starting FAST conversion...\n")
start = time.time()

train_out = os.path.join(ORIGINAL_DATA, 'instances_train_trashcan.json')
val_out = os.path.join(ORIGINAL_DATA, 'instances_val_trashcan.json')

fast_coco_convert(IMAGES_DIR, ANNOTATIONS_DIR, train_out, 'train', 0.8)
fast_coco_convert(IMAGES_DIR, ANNOTATIONS_DIR, val_out, 'val', 0.8)

print(f"\n⏱️  Time: {(time.time()-start)/60:.1f} minutes")
print("✅ Conversion complete!")

## Step 6: Check GPU

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  Enable GPU: Runtime → Change runtime type → T4")

## Step 7: Configure Training

In [ ]:
BATCH_SIZE = 8
EPOCHS = 50
LR = 0.01
SAVE_DIR = 'runs/train'

print(f"Batch: {BATCH_SIZE}, Epochs: {EPOCHS}, LR: {LR}")

## Step 8: 🚀 Train Model

In [ ]:
!python scripts/train.py \
  --data-dir "{ORIGINAL_DATA}" \
  --batch-size {BATCH_SIZE} \
  --epochs {EPOCHS} \
  --lr {LR} \
  --save-dir {SAVE_DIR}

## Step 9: View Results

In [ ]:
import glob

ckpt_dir = os.path.join(SAVE_DIR, 'checkpoints')
if os.path.exists(ckpt_dir):
    ckpts = glob.glob(os.path.join(ckpt_dir, '*.pt'))
    print(f"✅ {len(ckpts)} checkpoint(s) found:")
    for c in ckpts:
        size = os.path.getsize(c) / (1024*1024)
        print(f"   📦 {os.path.basename(c)} ({size:.1f} MB)")
else:
    print("⚠️  No checkpoints yet - training may still be running")
    print("   This is normal if training is in progress!")

## Step 10: Download Model

In [ ]:
from google.colab import files

best_model = os.path.join(SAVE_DIR, 'checkpoints', 'best_model.pt')
if os.path.exists(best_model):
    files.download(best_model)
    print("✅ Download started!")
else:
    print("❌ Model not found - training may not have completed")